In [ ]:
# Imports for VAE MNIST example
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


In [ ]:
#
# Encoder q(z|x)
#
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, z_dim):
        super(Encoder, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        # mean of latent distribution
        self.fc_mu = nn.Linear(hidden_dim, z_dim)
        # log-variance of latent distribution
        self.fc_logvar = nn.Linear(hidden_dim, z_dim)
    def forward(self, x):
        h = F.relu(self.fc1(x))
        mu = self.fc_mu(h)
        log_var = self.fc_logvar(h)
        return mu, log_var

In [ ]:
# Reparameterization trick
# z = mu + sigma * eps
#
def reparameterize(mu, log_var):
    # standard deviation
    std = torch.exp(0.5 * log_var)
    # sample epsilon ~ N(0, Ι)
    eps = torch.randn_like(std)
    # reparameterized sample
    return mu + eps * std

In [ ]:
# Decoder p(x|z)
class Decoder(nn.Module):
    def __init__(self, z_dim, hidden_dim, output_dim):
        super(Decoder, self).__init__()
        self.fc1 = nn.Linear(z_dim, hidden_dim)
        self.fc_out = nn.Linear(hidden_dim, output_dim)
    def forward(self, z):
        h = F.relu(self.fc1(z))
        # outputs in [0,1]
        x_recon = torch.sigmoid(self.fc_out(h))
        return x_recon

In [ ]:
#
# Variational Autoencoder
#
class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dim, z_dim):
        super(VAE, self).__init__()
        self.encoder = Encoder(input_dim, hidden_dim, z_dim)
        self.decoder = Decoder(z_dim, hidden_dim, input_dim)
    def forward(self, x):
        mu, log_var = self.encoder(x)
        z = reparameterize(mu, log_var)
        x_recon = self.decoder(z)
        return x_recon, mu, log_var
    def estimate_px_monte_carlo(self, x, num_samples=100):
        """
        Estimate p(x) for a batch of data x using the Monte Carlo estimator:
        p(x) ≈ (1/K) sum_k p_theta(x|z_k) p(z_k) / q_phi(z_k|x)
        Args:
            x: input batch (tensor)
            num_samples: number of importance samples (K)
        Returns:
            px: estimated p(x) for each sample in the batch (tensor)
        """
        self.eval()
        with torch.no_grad():
            mu, log_var = self.encoder(x)
            mu = mu.unsqueeze(1).expand(-1, num_samples, -1)
            log_var = log_var.unsqueeze(1).expand(-1, num_samples, -1)
            x_exp = x.unsqueeze(1).expand(-1, num_samples, -1)
            std = torch.exp(0.5 * log_var)
            eps = torch.randn_like(std)
            z = mu + eps * std
            x_recon = self.decoder(z)
            # log p(x|z) for Gaussian with fixed variance
            log_px_z = -((x_exp - x_recon) ** 2).sum(dim=-1)
            # log p(z) for standard normal
            log_pz = -0.5 * (z ** 2).sum(dim=-1)
            # log q(z|x) for diagonal Gaussian
            log_qz_x = -0.5 * (((z - mu) ** 2) / (std ** 2) + 2 * torch.log(std)).sum(dim=-1)
            # Importance weights
            log_weight = log_px_z + log_pz - log_qz_x
            # Monte Carlo estimator: p(x) ≈ (1/K) sum_k exp(log_weight_k)
            px = torch.exp(torch.logsumexp(log_weight, dim=1) - torch.log(torch.tensor(num_samples, dtype=log_weight.dtype, device=log_weight.device)))
        return px


In [ ]:
# Loss function
# ELBO = Reconstruction loss + KL divergence
#
def vae_loss(x, x_recon, mu, log_var):
    # Fixed variance sigma^2 = 0.5 (sigma = sqrt(2)/2)
    # For sigma^2=0.5, 2*sigma^2=1, so denominator is 1
    # Gaussian NLL (ignoring constant term) reduces to MSE sum
    recon_loss = F.mse_loss(x_recon, x, reduction="sum")
    # KL divergence between q(z|x) and p(z)
    kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
    return recon_loss + kl_loss


In [ ]:
# Hyperparameters
#
dim_input = 28 * 28 # Flattened MNIST image size
hidden_units = 128 # Hidden layer size
dim_z = 32 # Latent space dimension
batch_size = 256
epochs = 200
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Training setup
#
# Transform: convert to tensor and flatten to 784-dim vector
transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x.view(-1))])
train_dataset = datasets.MNIST(root="./data", train=True, transform=transform, download=True)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
vae = VAE(dim_input, hidden_units, dim_z).to(device)
optimizer = optim.Adam(vae.parameters(), lr=1e-3)

In [ ]:
# Training loop
#
for epoch in range(epochs):
    vae.train()
    train_loss = 0
    for x, in train_loader:
        x = x.to(device)
        optimizer.zero_grad()
        x_recon, mu, log_var = vae(x)
        loss = vae_loss(x, x_recon, mu, log_var)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {train_loss/len(train_loader.dataset):.4f}")


In [ ]:
#
# Testing / reconstruction
#
test_dataset = datasets.MNIST(root="./data", train=False, transform=transform, download=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

vae.eval()
with torch.no_grad():
    for x, in test_loader:
        x = x.to(device)
        x_recon, _, _ = vae(x)
        break
print("Reconstructed batch shape:", x_recon.shape)